# OpenPlaque — Secondary branch target-free continuation

Focused source-CCTA experiment. Starts from the last unequivocally good point on the accepted 13.8-mm secondary branch and searches forward without using the prior 17.9-mm relaunch as a target. The old relaunch is comparison-only. Research use only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse controls: True=reuse valid cache; False=force recompute/update.
REUSE_SOURCE_CT = True
REUSE_FROZEN_GEOMETRY = True
REUSE_CONTINUATION_SEARCH = False


In [ ]:
# Dependencies
!pip -q install scipy pandas matplotlib


In [ ]:
# Clone the fresh experiment branch (created directly from stable main baseline).
import os, shutil
if os.path.exists('/content/OpenPlaque'):
    shutil.rmtree('/content/OpenPlaque')
!git clone -q --depth 1 --branch secondary-target-free-continuation-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%cd /content/OpenPlaque
!git rev-parse HEAD


In [ ]:
# Initialize workflow
import sys
sys.path.insert(0, '/content/OpenPlaque/src')
from openplaque.secondary_target_free_continuation import SecondaryTargetFreeContinuationWorkflow
from openplaque.lcx_gap_bridge import arc_mm, orthogonal_plane

reuse = {
    'source_ct': REUSE_SOURCE_CT,
    'frozen_geometry': REUSE_FROZEN_GEOMETRY,
    'continuation_search': REUSE_CONTINUATION_SEARCH,
}
wf = SecondaryTargetFreeContinuationWorkflow(reuse=reuse)
display(wf.cache_status())


In [ ]:
# Load source CCTA and freeze accepted-branch source geometry.
# Verify search_target is None and inspect the old 17.9-mm return distance.
wf.load_source_ct()
geometry = wf.load_frozen_geometry()
display(geometry)


In [ ]:
# Target-free continuation search.
summary = wf.search_continuation(max_new_mm=10.0, beam_width=50)
display(summary)
if wf.candidates is not None and len(wf.candidates):
    display(wf.candidates.head(15))
else:
    print('No >=3.5 mm candidate survived for full serial QC.')


In [ ]:
# Critical diagnostic: unlike the flawed bridge run, a progressing search should have multiple step_index values.
if wf.diag is not None and len(wf.diag):
    step_summary = wf.diag.groupby('step_index').agg(
        proposals=('plane_score','size'),
        best_plane_score=('plane_score','max'),
        max_new_length_mm=('new_length_mm','max'),
        max_displacement_mm=('endpoint_displacement_mm','max'),
        max_old_branch_separation_mm=('old_branch_separation_mm','max'),
    ).reset_index()
    display(step_summary)
    print('Search steps reached:', int(step_summary.step_index.max()) + 1)
else:
    print('No step diagnostics generated.')


In [ ]:
# QC figures saved to Drive report folder.
import numpy as np, matplotlib.pyplot as plt
from IPython.display import display, Image

ref = np.vstack([wf.reference, wf.trunk])
p = wf.best_path

# Geometry comparison: accepted branch, candidate, and old relaunch (comparison only).
fig, axs = plt.subplots(1,3,figsize=(16,5))
for ax,(a,b,title) in zip(axs,[(1,2,'axial'),(0,2,'coronal'),(0,1,'sagittal')]):
    ax.plot(ref[:,b],ref[:,a],lw=1,label='LAD/trunk')
    ax.plot(wf.seed[:,b],wf.seed[:,a],lw=2,label='accepted secondary branch')
    if wf.prior_course is not None:
        ax.plot(wf.prior_course[:,b],wf.prior_course[:,a],lw=1,alpha=.6,label='prior relaunch (comparison)')
    ax.plot(p[:,b],p[:,a],lw=2.5,label='target-free candidate')
    ax.scatter([wf.source_point[b]],[wf.source_point[a]],s=40,label='fixed source')
    ax.set_title(title); ax.set_aspect('equal'); ax.legend(fontsize=7)
fig.tight_layout()
geom_png = wf.out/'01_target_free_geometry.png'
fig.savefig(geom_png,dpi=150); plt.close(fig)

# Orthogonal source-CCTA cross-sections along the best candidate.
fig, axs = plt.subplots(3,4,figsize=(13,10)); axs=axs.ravel(); used=0
if len(p) >= 2:
    s=arc_mm(p,wf.spacing)
    for ax,x in zip(axs,np.linspace(0,s[-1],min(12,max(2,len(p))))):
        i=int(np.argmin(abs(s-x))); i0=max(0,i-4); i1=min(len(p)-1,i+4)
        im,c=orthogonal_plane(wf.ct,p[i],(p[i1]-p[i0])*wf.spacing,wf.spacing)
        ax.imshow(im,cmap='gray',vmin=0,vmax=800,extent=[c[0],c[-1],c[-1],c[0]])
        ax.set_title(f'{s[i]:.1f} mm'); ax.set_xticks([]); ax.set_yticks([]); used+=1
for ax in axs[used:]: ax.axis('off')
fig.suptitle('Best target-free candidate — orthogonal source-CCTA planes')
fig.tight_layout()
xs_png = wf.out/'02_target_free_cross_sections.png'
fig.savefig(xs_png,dpi=150); plt.close(fig)

display(Image(filename=str(geom_png)))
display(Image(filename=str(xs_png)))


In [ ]:
# Package outputs back to Drive.
import json, zipfile
html = wf.out/'OPENPLAQUE_SECONDARY_TARGET_FREE_CONTINUATION_REPORT.html'
rows=''.join(f'<tr><th>{k}</th><td>{v}</td></tr>' for k,v in wf.summary.items())
html.write_text(
    '<html><body><h1>OpenPlaque Secondary Branch — Target-Free Continuation</h1>'
    f'<p><b>Status: {wf.summary.get("status")}</b></p><table border="1" cellpadding="4">{rows}</table>'
    '<h2>Geometry</h2><img src="01_target_free_geometry.png" width="95%">'
    '<h2>Cross-sections</h2><img src="02_target_free_cross_sections.png" width="95%">'
    '<p>Prior relaunch is comparison-only. No distal target is used. No LCX identity is assigned automatically.</p></body></html>'
)
zip_path = wf.out/'OPENPLAQUE_SECONDARY_TARGET_FREE_CONTINUATION_REPORT_BACK.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in wf.out.iterdir():
        if f.is_file() and f != zip_path:
            z.write(f,arcname=f.name)
print('Status:', wf.summary.get('status'))
print('Accepted continuation:', wf.summary.get('accepted_continuation'))
print('Report:', html)
print('Report-back ZIP:', zip_path)
